# S2F — Few-Shot Segmentation di PASCAL-5i (Colab)

Pipeline: sam encoder → protoformer → skip connection.

Urutan: **konfigurasi → git → install → unduh+ekstrak data → train 4 fold → evaluasi (mIoU + FB-IoU)**.

Data diunduh otomatis (sel 3) ke disk **lokal** Colab `<DATA_ROOT>` (ekstrak cepat),
sedangkan archive-nya di-cache di Drive `<ARCHIVE_DIR>` (jadi sesi berikutnya tinggal ekstrak).
Repo kode di-clone/reset dari git (sel 1b).

Edit semua path di sel **KONFIGURASI** di bawah.

## 1. Konfigurasi (ubah path di sini)

In [ ]:
# ================= KONFIGURASI =================
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR   = "/content/drive/MyDrive/project/S2F"                     # folder kode (repo ini)
REPO_URL   = "https://github.com/USER/REPO.git"               # WAJIB diisi
REPO_BRANCH= "master"                                          # branch terbaru
DATA_ROOT  = "/content/datasets"                                     # LOKAL colab (ekstrak cepat, ephemeral)
ARCHIVE_DIR= "/content/drive/MyDrive/project/datasets/_archives"     # cache archive di Drive
SPLIT_ROOT = "/content/drive/MyDrive/project/datasets/splits/pascal"  # hasil make_splits.py
CACHE_DIR  = "/content/drive/MyDrive/project/datasets/_cache"        # cache index kelas (drive)
HIERA_CKPT = "/content/drive/MyDrive/project/checkpoints/sam2_hiera_large.pt"
OUT_DIR    = "/content/drive/MyDrive/project/S2F/runs"

VOC_DIR = f"{DATA_ROOT}/VOC2012"
SBD_DIR = f"{DATA_ROOT}/SBD"

# ---------- protokol ----------
SHOT         = 1          # 1-shot
SIZE         = 352
FOLDS        = [0, 1, 2, 3]
EPOCHS       = 20
EPISODES     = 1000       # episode train per epoch
VAL_EPISODES = 1000       # episode validasi / test
BATCH_SIZE   = 4
LR           = 1e-3
NUM_WORKERS  = 4
AMP_DTYPE    = "auto"   # auto | bf16 | fp16 | none (auto: bf16 Ampere+, fp16 T4)

# ---------- lokasi data turunan ----------
SBD_IMG   = f"{SBD_DIR}/img"
SBD_CLS   = f"{SBD_DIR}/cls"
SBD_PNG   = f"{SBD_DIR}/cls_png"     # hasil prepare_sbd.py
VOC_IMG   = f"{VOC_DIR}/JPEGImages"
VOC_MASK  = f"{VOC_DIR}/SegmentationClass"
print("config ok")

## 1b. Ambil / perbarui kode dari git

First run: clone. Selanjutnya: fetch + `reset --hard origin/<branch>` ke commit terbaru.
Cukup re-run sel ini lalu sel training — tidak perlu reconnect Colab.
`reset --hard` tidak menyentuh file untracked/ignored (`runs/` aman).

In [ ]:
import os, subprocess

def git(*a):
    print("$", "git", *a)
    subprocess.run(["git", *a], check=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    if os.path.isdir(REPO_DIR) and os.listdir(REPO_DIR):
        print(f"{REPO_DIR} ada tapi bukan git repo -> lewati clone.")
        print("Pindahkan/rename dulu, atau set REPO_DIR ke folder kosong.")
    else:
        git("clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR)
else:
    git("-C", REPO_DIR, "fetch", "origin", REPO_BRANCH)
    git("-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}")

subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"])

## 2. Setup environment
Catatan: ekstensi CUDA SAM2 (`_C`) **tidak** dipakai oleh pipeline ini (hanya trunk encoder), jadi tidak perlu di-build.

In [ ]:
# torch sudah tersedia di Colab; install sisa dependensi
!pip -q install hydra-core==1.3.2 omegaconf imageio pysodmetrics tqdm scipy

In [ ]:
import os, sys
assert os.path.isdir(REPO_DIR), f"REPO_DIR tidak ada: {REPO_DIR}"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

# cek path data penting
for p in [SBD_IMG, VOC_IMG, VOC_MASK, HIERA_CKPT]:
    print("OK " if os.path.exists(p) else "MISSING ", p)

## 3. Unduh + siapkan data (first run)

`download_data.py` mengunduh VOC2012, SBD, dan checkpoint SAM2 kalau belum ada
(otomatis skip yang sudah ada). Lalu `prepare_sbd.py` + `make_splits.py`
menyiapkan mask PNG dan split fold.

In [ ]:
# unduh archive ke Drive (cache) + EKSTRAK ke disk lokal colab (cepat)
# VOC: ~2 GB, SBD: ~190 MB, ckpt: ~860 MB
!python download_data.py \
  --archive_dir "$ARCHIVE_DIR" \
  --extract_root "$DATA_ROOT" \
  --ckpt_path "$HIERA_CKPT"

In [ ]:
# konversi SBD .mat -> .png (lewati kalau SBD_PNG sudah ada)
if not os.path.isdir(SBD_PNG) or len(os.listdir(SBD_PNG)) == 0:
    !python prepare_sbd.py --cls_dir "$SBD_CLS" --out_dir "$SBD_PNG"
else:
    print("SBD_PNG sudah ada, lewati")

In [ ]:
# buat split PASCAL-5i (lewati kalau SPLIT_ROOT sudah berisi fold0..fold3)
if not os.path.isdir(f"{SPLIT_ROOT}/fold0"):
    !python make_splits.py \
      --sbd_img_dir "$SBD_IMG" --sbd_mask_dir "$SBD_PNG" \
      --voc_img_dir "$VOC_IMG" --voc_mask_dir "$VOC_MASK" \
      --out_dir "$SPLIT_ROOT"
else:
    print("split sudah ada, lewati")

## 4. Training (semua fold, 1-shot)

In [ ]:
import subprocess

for fold in FOLDS:
    print("=" * 60); print(f"TRAIN fold {fold}"); print("=" * 60)
    cmd = [
        "python", "train_fss.py",
        "--fold", str(fold),
        "--hiera_path", HIERA_CKPT,
        "--train_img_dir", SBD_IMG,
        "--train_mask_dir", SBD_PNG,
        "--train_split", f"{SPLIT_ROOT}/fold{fold}/train.txt",
        "--val_img_dir", VOC_IMG,
        "--val_mask_dir", VOC_MASK,
        "--val_split", f"{SPLIT_ROOT}/fold{fold}/val.txt",
        "--save_path", f"{OUT_DIR}/fold{fold}",
        "--shot", str(SHOT), "--size", str(SIZE),
        "--epochs", str(EPOCHS),
        "--episodes_per_epoch", str(EPISODES),
        "--val_episodes", str(VAL_EPISODES),
        "--batch_size", str(BATCH_SIZE), "--lr", str(LR),
        "--num_workers", str(NUM_WORKERS),
        "--cache_dir", CACHE_DIR,
        "--amp_dtype", AMP_DTYPE,
    ]
    subprocess.run(cmd, check=True)

## 5. Evaluasi + agregasi 4 fold (mIoU, FB-IoU)

In [ ]:
import subprocess, re
import numpy as np

rows = []
for fold in FOLDS:
    ckpt = f"{OUT_DIR}/fold{fold}/S2F-fold{fold}-best.pth"
    if not os.path.exists(ckpt):
        ckpt = f"{OUT_DIR}/fold{fold}/S2F-fold{fold}-last.pth"
    print("\n" + "=" * 60); print(f"TEST fold {fold}  ({ckpt})"); print("=" * 60)
    out = subprocess.run([
        "python", "test_fss.py",
        "--fold", str(fold),
        "--checkpoint", ckpt,
        "--img_dir", VOC_IMG,
        "--mask_dir", VOC_MASK,
        "--split", f"{SPLIT_ROOT}/fold{fold}/val.txt",
        "--shot", str(SHOT), "--size", str(SIZE),
        "--episodes", str(VAL_EPISODES),
        "--num_workers", str(NUM_WORKERS),
        "--cache_dir", CACHE_DIR,
        "--amp_dtype", AMP_DTYPE,
    ], capture_output=True, text=True)
    print(out.stdout)
    if out.returncode != 0:
        print(out.stderr[-2000:])
    m = re.search(r"mIoU\s*:\s*([\d.]+)", out.stdout)
    f = re.search(r"FB-IoU\s*:\s*([\d.]+)", out.stdout)
    if m and f:
        rows.append((fold, float(m.group(1)), float(f.group(1))))

print("\n================ HASIL AKHIR ================")
for fold, miou, fb in rows:
    print(f"fold {fold}: mIoU {miou:.4f}  FB-IoU {fb:.4f}")
if rows:
    print(f"mean (4 fold): mIoU {np.mean([r[1] for r in rows]):.4f}  "
          f"FB-IoU {np.mean([r[2] for r in rows]):.4f}")

## 6. (Opsional) uji 1 fold saja + simpan PNG

Berguna untuk debug cepat sebelum loop penuh.

In [ ]:
FOLD = 0
!python test_fss.py \
  --fold $FOLD \
  --checkpoint "$OUT_DIR/fold$FOLD/S2F-fold$FOLD-best.pth" \
  --img_dir "$VOC_IMG" --mask_dir "$VOC_MASK" \
  --split "$SPLIT_ROOT/fold$FOLD/val.txt" \
  --shot $SHOT --size $SIZE --episodes 200 \
  --num_workers $NUM_WORKERS --amp_dtype $AMP_DTYPE --cache_dir "$CACHE_DIR" \
  --save_png_dir "$OUT_DIR/fold$FOLD/preds"